In [29]:
!pip install deepface
import pandas as pd
from datetime import datetime
import os
import csv

In [2]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [36]:
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected

LOG_FILE = "engagement_logs.csv"

if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["Timestamp", "Name", "Total_Score"])

try:
  # Take a photo.
  image_file = take_photo()

  # Detect faces and estimate emotions.
  result = DeepFace.analyze(img_path = image_file, actions = ['emotion'])

  # Print the number of persons detected.
  print('%d person(s) detected.' % len(result))

  try:
      # Look for the cropped face in your database
      db_results = db_results = DeepFace.find(
        img_path=image_file,
        db_path="allowed_faces/",
        model_name="Facenet512",
        detector_backend="opencv",
        enforce_detection=False
      )
      if len(db_results) > 0 and not db_results[0].empty:
          #print(db_results[0])
          # Extract name from the file path (e.g., "allowed_faces/luca.jpg" -> "luca")
          matched_path = db_results[0]['identity'][0]
          person_name = matched_path.split('/')[-1].split('.')[0].capitalize()
      else:
          person_name = "Unknown Guest"
  except:
    person_name = "Unknown Guest"

  print(f"--- {person_name} ---")

  # 【追加】感情ごとの点数（重み付け）の設定
  emotion_scores = {
      'happy': 5,
      'surprise': 4,
      'neutral': 3,
      'sad': 2,
      'fear': 2,
      'disgust': 1,
      'angry': 1
  }

  session_logs = []

  # Print the probability of each emotion for each person.
  for i in range(len(result)):
    print('--- Person #%d ---' % (i + 1))

    total_score = 0.0  # 【追加】この人の総合評価点を初期化

    for e, p in result[i]['emotion'].items():
      # e は感情名（文字列）、p はその確率（0〜100のパーセンテージ）
      print('%s: %f%%' % (e, p))

      # 【追加】定義した点数を取得（万が一想定外のキーがあれば0点にする）
      score = emotion_scores.get(e, 0)

      # 【追加】「確率 × 点数」を計算して合計に加算
      # ※DeepFaceの確率(p)は0〜100%で返るため、割合(0〜1)に直す場合は p / 100 にします。
      # ここでは「割合(0〜1) × 点数」として計算しています。
      total_score += (p / 100.0) * score

    # 【追加】この人の総合評価点（積の合算）を表示（小数点以下2桁まで）
    print('=> Total Score: %.2f / 5.00' % total_score)

    # Get the current timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Save this person's record temporarily
    session_logs.append([timestamp, person_name, round(total_score, 2)])

    # Write all session records to the CSV file at once
    with open(LOG_FILE, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(session_logs)
        print(f"Successfully logged {len(session_logs)} entry/entries to {LOG_FILE}!")

except FaceNotDetected:
  # If no face is detected, a FaceNotDetected exception is raised.
  print('No face detected.')

except Exception as e:
  # If any other exceptions are raised, details will be displayed.
  print(e)

<IPython.core.display.Javascript object>

1 person(s) detected.
26-06-03 02:33:29 - Searching photo.jpg in 3 length datastore
26-06-03 02:33:30 - find function duration 0.7920670509338379 seconds
--- Komakiharuhito ---
--- Person #1 ---
angry: 0.780027%
disgust: 0.000035%
fear: 0.099319%
happy: 0.083615%
sad: 2.002754%
surprise: 0.002537%
neutral: 97.031715%
=> Total Score: 2.97 / 5.00
Successfully logged 1 entry/entries to engagement_logs.csv!
